In [1]:
import pandas as pd
import numpy as np

In [2]:
customers = pd.read_csv('customers.csv')
order_items = pd.read_csv('order_items.csv')
orders = pd.read_csv('orders.csv')
products = pd.read_csv('products.csv')
payments = pd.read_csv('payments.csv')

In [3]:
# 1. Sabse pehle orders aur order_items ko merge karo (order_id ke basis par)
df = pd.merge(orders, order_items, on='order_id', how='inner')

# 2. Agar product details (category name etc.) chahiye, toh products ke sath merge karo (product_id ke basis par)
df = pd.merge(df, products, on='product_id', how='left')

# 3. Check karne ke liye ki data kaisa dikh raha hai aur columns ke sahi naam kya hain
print("Merged Data Shape:", df.shape)
df.head()

Merged Data Shape: (110189, 13)


,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02,2017-10-10,2017-10-18,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06,29.99,8.72,utilidades_domesticas
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24,2018-08-07,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30,118.70,22.76,perfumaria
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08,2018-08-17,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13,159.90,19.22,automotivo
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18,2017-12-02,2017-12-15,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23,45.00,27.20,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13,2018-02-16,2018-02-26,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19,19.90,8.72,papelaria


In [4]:
# 1. Date column ko string se datetime format me convert karo
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

# 2. Date se naye features (Year aur Month) extract karo
df['year'] = df['order_purchase_timestamp'].dt.year
df['month'] = df['order_purchase_timestamp'].dt.month

# 3. Data ko Monthly Sales ke format me aggregate (Group By) karo
# Hamein har mahine ki total sales (sum of price) chahiye
monthly_sales = df.groupby(['year', 'month'])['price'].sum().reset_index()

# Columns ka naam clean kar dete hain
monthly_sales.columns = ['year', 'month', 'total_sales']

# Data ko purane se naye mahine ke sequence me sort kar lo
monthly_sales = monthly_sales.sort_values(by=['year', 'month']).reset_index(drop=True)

# 4. Model ke liye ek continuous 'Time Index' lagao (Month 1, Month 2, Month 3...)
monthly_sales['time_index'] = range(1, len(monthly_sales) + 1)

# 5. Lag Feature banao (Pichle mahine ki sales kya thi)
monthly_sales['sales_last_month'] = monthly_sales['total_sales'].shift(1)

# Pehle mahine ke liye pichla mahina nahi hoga (NaN), toh us row ko hata dete hain
monthly_sales.dropna(inplace=True)

# 6. Final ML-Ready Data check karo
print("Feature Engineering complete! Dataset ka structure:")
monthly_sales.head()

Feature Engineering complete! Dataset ka structure:


,year,month,total_sales,time_index,sales_last_month
1,2016,10,40325.11,2,134.97
2,2016,12,10.90,3,40325.11
3,2017,1,111798.36,4,10.90
4,2017,2,234223.40,5,111798.36
5,2017,3,359198.85,6,234223.40


In [6]:
monthly_sales.to_csv('ml_ready_stat_data.csv', index=False)